# Pipeline 03 Spectral Feature and Pair Validation

This notebook validates the spectral feature shards generated by `03_extract_spectral_features.py` for the new full BigCloneBench Type-3 benchmark.

It focuses on three questions:

- Do train-pair methods have spectral features for the expected graph layers?
- Do Type-3 clone pairs and non-clone pairs show different spectral similarity distributions?
- Do the code, graph structure, and eigenvalue spectrum look consistent when inspected side by side?

In [ ]:
from pathlib import Path
import json
import os
import pickle
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "pipelines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BCB_CLONE_TYPE = "3"
BENCH_DIR = PROJECT_ROOT / "bench_data" / "bcb_full_type3"
OUTPUT_ROOT = PROJECT_ROOT.parent / "outputs" / "type3"
CLEAN_GRAPHS_DIR = OUTPUT_ROOT / "clean_graphs"
SPECTRAL_DIR = OUTPUT_ROOT / "spectral_features"

DATA_PATH = BENCH_DIR / "data.jsonl"
TRAIN_PATH = BENCH_DIR / "train.txt"
GRAPH_MANIFEST_PATH = CLEAN_GRAPHS_DIR / "graph_shards_manifest.json"
SPECTRAL_MANIFEST_PATH = SPECTRAL_DIR / "spectral_features_manifest.json"

GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
RANDOM_SEED = 42
PAIR_SAMPLE_PER_LABEL = 500
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

paths = {
    "data_jsonl": DATA_PATH,
    "train_txt": TRAIN_PATH,
    "graph_manifest": GRAPH_MANIFEST_PATH,
    "spectral_manifest": SPECTRAL_MANIFEST_PATH,
}
pd.DataFrame([{"name": k, "path": str(v), "exists": v.exists()} for k, v in paths.items()])

## 1. Load Manifests and Train Pairs

If the spectral manifest does not exist, run pipeline `03` first.

In [ ]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)

def load_train_pairs(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            left, right, label = line.split("\t")[:3]
            rows.append((str(left), str(right), int(label)))
    return pd.DataFrame(rows, columns=["left_id", "right_id", "label"])

graph_manifest = load_json(GRAPH_MANIFEST_PATH)
spectral_manifest = load_json(SPECTRAL_MANIFEST_PATH)
train_df = load_train_pairs(TRAIN_PATH)

summary = {
    "graph_total_methods": graph_manifest.get("total_methods"),
    "graph_num_shards": len(graph_manifest.get("shards", [])),
    "spectral_total_methods": spectral_manifest.get("total_methods"),
    "spectral_num_shards": len(spectral_manifest.get("shards", [])),
    "spectral_mode": spectral_manifest.get("mode"),
    "train_pairs": len(train_df),
    "train_unique_methods": len(set(train_df.left_id).union(train_df.right_id)),
}
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(train_df["label"].value_counts().rename_axis("label").reset_index(name="count"))

## 2. Lazy Loaders

These helpers load only requested method IDs from `data.jsonl`, graph shards, and spectral feature shards. This keeps the notebook usable with the full dataset.

In [ ]:
def iter_data_jsonl():
    with DATA_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def load_code_records(method_ids):
    wanted = {str(mid) for mid in method_ids}
    found = {}
    for record in iter_data_jsonl():
        method_id = str(record["idx"])
        if method_id in wanted:
            found[method_id] = record.get("func", "")
            if len(found) == len(wanted):
                break
    return found

def load_from_pickle_shards(shard_paths, method_ids, desc):
    wanted = {str(mid) for mid in method_ids}
    found = {}
    for shard_path in tqdm(shard_paths, desc=desc, unit="shard"):
        with Path(shard_path).open("rb") as f:
            shard = pickle.load(f)
        for method_id in list(wanted):
            if method_id in shard:
                found[method_id] = shard[method_id]
                wanted.remove(method_id)
        if not wanted:
            break
    return found

def load_graph_records(method_ids):
    return load_from_pickle_shards(graph_manifest["shards"], method_ids, "Loading graph shards")

def load_feature_records(method_ids):
    return load_from_pickle_shards(spectral_manifest["shards"], method_ids, "Loading feature shards")

display(Markdown("Lazy loaders are ready."))

## 3. Feature Coverage for Sampled Train Pairs

This checks whether sampled clone and non-clone pairs have non-empty eigenvalue arrays for each graph type.

In [ ]:
sample_pos = train_df[train_df.label == 1].sample(min(PAIR_SAMPLE_PER_LABEL, int((train_df.label == 1).sum())), random_state=RANDOM_SEED)
sample_neg = train_df[train_df.label == 0].sample(min(PAIR_SAMPLE_PER_LABEL, int((train_df.label == 0).sum())), random_state=RANDOM_SEED)
pair_sample = pd.concat([sample_pos, sample_neg], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
sample_method_ids = set(pair_sample.left_id).union(pair_sample.right_id)

features = load_feature_records(sample_method_ids)

coverage_rows = []
for gtype in GRAPH_TYPES:
    methods_with_layer = 0
    methods_with_nonempty_eigs = 0
    eig_lengths = []
    for method_id in sample_method_ids:
        layer = features.get(method_id, {}).get(gtype, {})
        eigs = np.asarray(layer.get("eigenvalues", []), dtype=float)
        if gtype in features.get(method_id, {}):
            methods_with_layer += 1
        if eigs.size > 0:
            methods_with_nonempty_eigs += 1
            eig_lengths.append(eigs.size)
    coverage_rows.append({
        "graph_type": gtype,
        "sample_methods": len(sample_method_ids),
        "methods_with_layer": methods_with_layer,
        "methods_with_nonempty_eigs": methods_with_nonempty_eigs,
        "nonempty_coverage": methods_with_nonempty_eigs / max(1, len(sample_method_ids)),
        "eigen_len_mean": float(np.mean(eig_lengths)) if eig_lengths else 0.0,
        "eigen_len_median": float(np.median(eig_lengths)) if eig_lengths else 0.0,
        "eigen_len_max": int(np.max(eig_lengths)) if eig_lengths else 0,
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

## 4. PSS Similarity Distribution on Sampled Pairs

This gives an early visual signal about separability. If Type-3 clones and non-clones overlap heavily for every graph type, either the representation is not discriminative enough or there is a data/feature issue.

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from spectral_code.similarity.pss import PSSSimilarity

pss = PSSSimilarity()

def eigenvalues(method_id, graph_type):
    layer = features.get(str(method_id), {}).get(graph_type, {})
    return np.asarray(layer.get("eigenvalues", []), dtype=float)

score_rows = []
for row in tqdm(pair_sample.itertuples(index=False), total=len(pair_sample), desc="Scoring sampled pairs", unit="pair"):
    for gtype in GRAPH_TYPES:
        left = eigenvalues(row.left_id, gtype)
        right = eigenvalues(row.right_id, gtype)
        has_score = left.size > 0 and right.size > 0
        score = float(pss.compute(left, right)) if has_score else np.nan
        score_rows.append({
            "left_id": row.left_id,
            "right_id": row.right_id,
            "label": row.label,
            "graph_type": gtype,
            "pss": score,
            "status": "ok" if has_score else "missing_or_empty_eigenvalues",
            "left_eigs": left.size,
            "right_eigs": right.size,
        })

scores_df = pd.DataFrame(score_rows)
valid_scores_df = scores_df[scores_df.status == "ok"].copy()
missing_scores_df = scores_df[scores_df.status != "ok"].copy()

display(Markdown("### Valid PSS score summary"))
display(valid_scores_df.groupby(["graph_type", "label"])["pss"].describe())

display(Markdown("### Missing or empty eigenvalue summary"))
display(missing_scores_df.groupby(["graph_type", "label"]).size().rename("missing_pair_count").reset_index())
if not missing_scores_df.empty:
    display(Markdown("**These rows are pipeline-output failures, not low PSS scores. Re-run pipelines after fixing extraction before final threshold tuning.**"))
    display(missing_scores_df.head(20))

def plot_score_hist(ax, values, label, bins=40):
    values = pd.Series(values).dropna().astype(float)
    if values.empty:
        ax.text(0.5, 0.5, f"No valid {label} scores", ha="center", va="center", transform=ax.transAxes)
        return
    bin_edges = np.linspace(0.0, 1.0, bins + 1)
    if values.nunique() == 1:
        ax.axvline(values.iloc[0], linewidth=2.5, alpha=0.75, label=f"{label} (constant={values.iloc[0]:.4f}, n={len(values)})")
        return
    weights = np.ones(len(values)) / len(values)
    ax.hist(values, bins=bin_edges, weights=weights, alpha=0.55, label=f"{label} (n={len(values)})")

fig, axes = plt.subplots(len(GRAPH_TYPES), 1, figsize=(12, 4 * len(GRAPH_TYPES)), sharex=True)
for ax, gtype in zip(axes, GRAPH_TYPES):
    subset = valid_scores_df[valid_scores_df.graph_type == gtype]
    plot_score_hist(ax, subset[subset.label == 0].pss, label="non-clone label 0")
    plot_score_hist(ax, subset[subset.label == 1].pss, label=f"type-{BCB_CLONE_TYPE} clone label 1")
    ax.set_title(f"PSS distribution for {gtype.upper()}")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylabel("fraction of class")
    ax.grid(alpha=0.25)
    ax.legend()
axes[-1].set_xlabel("PSS similarity")
plt.tight_layout()
plt.show()

## 5. Code, Graph, and Spectrum Side by Side

The next helper shows a pair's source code, selected graph layer, eigenvalue curves, and PSS score in one place.

In [ ]:
def code_html(code):
    escaped = code.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return f"<pre style='white-space: pre-wrap; font-size: 12px; line-height: 1.35; border: 1px solid #ddd; padding: 12px; max-height: 460px; overflow: auto'>{escaped}</pre>"

def node_label(graph, node):
    data = graph.nodes[node]
    ntype = str(data.get("type", ""))[:22]
    label = str(data.get("label", ""))[:22]
    return ntype if not label or label == ntype else f"{ntype}\n{label}"

def draw_graph(graph, ax, title, max_nodes=70):
    if graph is None or graph.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "empty or missing graph", ha="center", va="center")
        ax.set_title(title)
        ax.axis("off")
        return
    display_graph = graph
    if graph.number_of_nodes() > max_nodes:
        display_graph = graph.subgraph(list(graph.nodes())[:max_nodes]).copy()
        title = f"{title} (first {max_nodes}/{graph.number_of_nodes()} nodes)"
    pos = nx.spring_layout(display_graph, seed=RANDOM_SEED, k=0.9)
    labels = {node: node_label(display_graph, node) for node in display_graph.nodes()}
    nx.draw_networkx_edges(display_graph, pos, ax=ax, alpha=0.35, arrows=True, arrowsize=8, width=0.8)
    nx.draw_networkx_nodes(display_graph, pos, ax=ax, node_size=420, node_color="#fdd49e", edgecolors="#8c2d04", linewidths=0.7)
    nx.draw_networkx_labels(display_graph, pos, labels=labels, ax=ax, font_size=6)
    ax.set_title(title)
    ax.axis("off")

def plot_spectrum(ax, eigs, title, max_points=250):
    eigs = np.asarray(eigs, dtype=float)
    if eigs.size == 0:
        ax.text(0.5, 0.5, "empty eigenvalue vector", ha="center", va="center")
        ax.set_title(title)
        return
    values = eigs[:max_points]
    ax.plot(np.arange(values.size), values, linewidth=1.4)
    ax.set_title(f"{title} | len={eigs.size}")
    ax.set_xlabel("index")
    ax.set_ylabel("eigenvalue")
    ax.grid(alpha=0.25)

def show_pair_deep(row, graph_type="ast", max_graph_nodes=70, max_spectrum_points=250):
    left_id = str(row.left_id)
    right_id = str(row.right_id)
    label = int(row.label)
    code_map = load_code_records([left_id, right_id])
    graph_map = load_graph_records([left_id, right_id])
    feature_map = load_feature_records([left_id, right_id])

    left_eigs = np.asarray(feature_map.get(left_id, {}).get(graph_type, {}).get("eigenvalues", []), dtype=float)
    right_eigs = np.asarray(feature_map.get(right_id, {}).get(graph_type, {}).get("eigenvalues", []), dtype=float)
    score = 0.0 if left_eigs.size == 0 or right_eigs.size == 0 else float(pss.compute(left_eigs, right_eigs))

    display(Markdown(f"### Pair `{left_id}` vs `{right_id}` | label={label} | {graph_type.upper()} PSS={score:.6f}"))
    display(HTML(
        "<div style='display:grid; grid-template-columns:1fr 1fr; gap:16px'>"
        f"<div><h4>Left: {left_id}</h4>{code_html(code_map.get(left_id, '<missing>'))}</div>"
        f"<div><h4>Right: {right_id}</h4>{code_html(code_map.get(right_id, '<missing>'))}</div>"
        "</div>"
    ))

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    draw_graph(graph_map.get(left_id, {}).get(graph_type), axes[0, 0], f"Left {graph_type.upper()} graph", max_nodes=max_graph_nodes)
    draw_graph(graph_map.get(right_id, {}).get(graph_type), axes[0, 1], f"Right {graph_type.upper()} graph", max_nodes=max_graph_nodes)
    plot_spectrum(axes[1, 0], left_eigs, f"Left {graph_type.upper()} spectrum", max_points=max_spectrum_points)
    plot_spectrum(axes[1, 1], right_eigs, f"Right {graph_type.upper()} spectrum", max_points=max_spectrum_points)
    plt.tight_layout()
    plt.show()

display(Markdown("Deep inspection helper loaded."))

## 6. Inspect Representative Pairs

This cell shows one Type-3 clone pair and one non-clone pair. Change `graph_type` to `cfg`, `ddg`, `pdg`, or `cpg` to inspect other representations.

In [ ]:
graph_type = "ast"

positive_row = train_df[train_df.label == 1].sample(1, random_state=RANDOM_SEED).iloc[0]
negative_row = train_df[train_df.label == 0].sample(1, random_state=RANDOM_SEED).iloc[0]

show_pair_deep(positive_row, graph_type=graph_type, max_graph_nodes=70)
show_pair_deep(negative_row, graph_type=graph_type, max_graph_nodes=70)

## 7. Inspect Extremes from the Sample

These examples are useful for debugging: low-scoring Type-3 clones may reveal graph extraction issues, while high-scoring non-clones may reveal PSS saturation or overly generic graph spectra.

In [ ]:
inspect_graph_type = "ast"
subset = scores_df[scores_df.graph_type == inspect_graph_type].copy()
valid_subset = subset[subset.status == "ok"].copy()
missing_subset = subset[subset.status != "ok"].copy()

lowest_clone = valid_subset[valid_subset.label == 1].sort_values("pss", ascending=True).head(1)
highest_nonclone = valid_subset[valid_subset.label == 0].sort_values("pss", ascending=False).head(1)

display(Markdown(f"### Missing/empty examples for `{inspect_graph_type.upper()}`"))
display(missing_subset.head(10))

display(Markdown(f"### Lowest valid-scoring Type-3 clone in sampled pairs for `{inspect_graph_type.upper()}`"))
display(lowest_clone)
if not lowest_clone.empty:
    show_pair_deep(lowest_clone.iloc[0], graph_type=inspect_graph_type, max_graph_nodes=70)

display(Markdown(f"### Highest valid-scoring non-clone in sampled pairs for `{inspect_graph_type.upper()}`"))
display(highest_nonclone)
if not highest_nonclone.empty:
    show_pair_deep(highest_nonclone.iloc[0], graph_type=inspect_graph_type, max_graph_nodes=70)

## Missing Feature Diagnostics

Rows marked `missing_or_empty_eigenvalues` are not real low-similarity examples. This diagnostic checks whether the graph itself is also missing. If both graph and spectrum are empty, the issue happened before spectral extraction, usually in pipeline `01` Joern parsing/exporting or DOT-to-method mapping.

In [ ]:
def diagnose_missing_rows(graph_type="ast", n=5):
    missing = scores_df[(scores_df.graph_type == graph_type) & (scores_df.status != "ok")].head(n)
    if missing.empty:
        display(Markdown(f"No missing examples found for `{graph_type.upper()}` in the sampled pairs."))
        return pd.DataFrame()

    method_ids = set(missing.left_id).union(set(missing.right_id))
    graph_map = load_graph_records(method_ids)
    feature_map = load_feature_records(method_ids)

    rows = []
    for method_id in sorted(method_ids):
        graph = graph_map.get(method_id, {}).get(graph_type)
        eigs = np.asarray(feature_map.get(method_id, {}).get(graph_type, {}).get("eigenvalues", []), dtype=float)
        rows.append({
            "method_id": method_id,
            "graph_type": graph_type,
            "graph_loaded": graph is not None,
            "graph_nodes": graph.number_of_nodes() if graph is not None else 0,
            "graph_edges": graph.number_of_edges() if graph is not None else 0,
            "eigenvalue_count": eigs.size,
            "likely_stage": "pipeline_01_or_02" if graph is None or graph.number_of_nodes() == 0 else "pipeline_03",
        })
    return pd.DataFrame(rows)

diagnose_missing_rows("ast", n=5)

## Interpretation Notes

- Type-3 clone PSS can vary substantially because Type-3 clones may preserve behavior while changing syntax and structure.
- A previous `0.0` score often meant missing eigenvalues, not real dissimilarity. The notebook now stores those cases as `NaN` and marks them with `status = missing_or_empty_eigenvalues`.
- Empty DDG/PDG can be normal for simple methods. Empty AST is suspicious and should be investigated before trusting final threshold results.
- If many valid non-clones still get PSS near `1.0`, then the scoring formula or graph spectrum may be saturating and we should debug PSS separately.

## Non-Clone Pairs With Perfect PSS

This section searches the full `train.txt` for label-0 pairs whose PSS is effectively `1.0`. These are the most important false-positive examples to inspect. They may indicate duplicate/non-clone labeling noise, overly generic spectra, or a bug in feature extraction/scoring.

In [ ]:
def load_features_for_pairs(pairs_df):
    method_ids = set(pairs_df.left_id).union(set(pairs_df.right_id))
    return load_feature_records(method_ids)

def score_pairs_with_features(pairs_df, graph_types=GRAPH_TYPES, tolerance=1e-12):
    local_features = load_features_for_pairs(pairs_df)
    rows = []
    for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df), desc="Scoring pairs", unit="pair"):
        for gtype in graph_types:
            left_layer = local_features.get(str(row.left_id), {}).get(gtype, {})
            right_layer = local_features.get(str(row.right_id), {}).get(gtype, {})
            left = np.asarray(left_layer.get("eigenvalues", []), dtype=float)
            right = np.asarray(right_layer.get("eigenvalues", []), dtype=float)
            has_score = left.size > 0 and right.size > 0
            score = float(pss.compute(left, right)) if has_score else np.nan
            rows.append({
                "left_id": str(row.left_id),
                "right_id": str(row.right_id),
                "label": int(row.label),
                "graph_type": gtype,
                "pss": score,
                "is_perfect": bool(has_score and score >= 1.0 - tolerance),
                "left_eigs": int(left.size),
                "right_eigs": int(right.size),
                "left_status": left_layer.get("status"),
                "right_status": right_layer.get("status"),
            })
    return pd.DataFrame(rows), local_features

# Full scan over all non-clone train pairs. This can take a little while, but it is much cheaper than rerunning pipelines.
nonclone_pairs_df = train_df[train_df.label == 0].copy()
nonclone_scores_df, nonclone_features = score_pairs_with_features(nonclone_pairs_df)

perfect_nonclone_df = nonclone_scores_df[nonclone_scores_df.is_perfect].copy()
display(Markdown("### Perfect-PSS non-clone count by graph"))
display(perfect_nonclone_df.groupby("graph_type").size().rename("perfect_nonclone_pairs").reset_index())

display(Markdown("### Example perfect-PSS non-clone pairs"))
display(perfect_nonclone_df.sort_values(["graph_type", "left_id", "right_id"]).head(30))

In [ ]:
def show_perfect_nonclone_examples(graph_type="ast", n=3, max_graph_nodes=70):
    examples = perfect_nonclone_df[perfect_nonclone_df.graph_type == graph_type].head(n)
    if examples.empty:
        display(Markdown(f"No perfect-PSS non-clone examples found for `{graph_type.upper()}`."))
        return
    for row in examples.itertuples(index=False):
        show_pair_deep(row, graph_type=graph_type, max_graph_nodes=max_graph_nodes)

show_perfect_nonclone_examples(graph_type="ast", n=3, max_graph_nodes=70)

In [ ]:
def compare_code_text_for_perfect_nonclones(graph_type="ast", n=20):
    examples = perfect_nonclone_df[perfect_nonclone_df.graph_type == graph_type].head(n)
    ids = set(examples.left_id).union(set(examples.right_id))
    code_map = load_code_records(ids)
    rows = []
    for row in examples.itertuples(index=False):
        left_code = code_map.get(str(row.left_id), "")
        right_code = code_map.get(str(row.right_id), "")
        rows.append({
            "left_id": row.left_id,
            "right_id": row.right_id,
            "graph_type": graph_type,
            "pss": row.pss,
            "exact_code_equal": left_code.strip() == right_code.strip(),
            "left_chars": len(left_code),
            "right_chars": len(right_code),
            "left_first_line": left_code.strip().splitlines()[0] if left_code.strip() else "",
            "right_first_line": right_code.strip().splitlines()[0] if right_code.strip() else "",
        })
    return pd.DataFrame(rows)

display(compare_code_text_for_perfect_nonclones(graph_type="ast", n=30))